In [ ]:
import numpy as np
import cv2
import pywt
import rasterio

from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.termination import get_termination

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

In [ ]:
def load_images(pan_path, ms_path):

    with rasterio.open(pan_path) as src:
        pan = src.read(1)

    with rasterio.open(ms_path) as src:
        ms = src.read()

    ms = np.transpose(ms, (1,2,0))

    return pan.astype(np.float32), ms.astype(np.float32)

In [ ]:
def awp_fusion(pan, ms, wavelet, level, alpha):

    fused = np.zeros_like(ms)

    coeffs = pywt.wavedec2(pan, wavelet=wavelet, level=level)

    detail_coeffs = coeffs[1:]

    for b in range(ms.shape[2]):

        band = ms[:,:,b]

        band_resized = cv2.resize(band, pan.shape[::-1])

        fused_band = band_resized.copy()

        for d in detail_coeffs:
            cH, cV, cD = d
            detail = (cH + cV + cD)/3
            detail = cv2.resize(detail, pan.shape[::-1])
            fused_band = fused_band + alpha * detail

        fused[:,:,b] = fused_band

    return fused

In [ ]:
def rmse(a,b):
    return np.sqrt(np.mean((a-b)**2))


def ergas(ms, fused):

    bands = ms.shape[2]
    h,l = ms.shape[0],ms.shape[1]

    ratio = 1

    s = 0

    for b in range(bands):
        rmse_val = rmse(ms[:,:,b], fused[:,:,b])
        mean_val = np.mean(ms[:,:,b])
        s += (rmse_val/mean_val)**2

    return 100/ratio * np.sqrt(s/bands)


def sam(ms, fused):

    dot = np.sum(ms*fused, axis=2)
    norm1 = np.linalg.norm(ms, axis=2)
    norm2 = np.linalg.norm(fused, axis=2)

    angle = np.arccos(dot/(norm1*norm2 + 1e-10))

    return np.mean(angle)


def spatial_corr(pan, fused):

    corr = []

    for b in range(fused.shape[2]):
        c = np.corrcoef(pan.flatten(), fused[:,:,b].flatten())[0,1]
        corr.append(c)

    return np.mean(corr)

In [ ]:
wavelets = ["haar","db2","db4","sym4"]

class FusionOptimization(Problem):

    def __init__(self, pan, ms):

        super().__init__(
            n_var=3,
            n_obj=6,
            n_constr=0,
            xl=np.array([0,0,1]),
            xu=np.array([1,3,4])
        )

        self.pan = pan
        self.ms = ms

    def _evaluate(self, X, out, *args, **kwargs):

        F = []

        for x in X:

            alpha = x[0]
            wavelet = wavelets[int(x[1])]
            level = int(x[2])

            fused = awp_fusion(self.pan, self.ms, wavelet, level, alpha)

            fused = np.clip(fused,0,255)

            o1 = ergas(self.ms,fused)
            o2 = sam(self.ms,fused)

            ssim_val = ssim(self.ms[:,:,0], fused[:,:,0], data_range=255)
            psnr_val = psnr(self.ms[:,:,0], fused[:,:,0], data_range=255)

            sc = spatial_corr(self.pan,fused)

            rmse_val = rmse(self.ms[:,:,0],fused[:,:,0])

            F.append([
                o1,
                o2,
                -ssim_val,
                -psnr_val,
                -sc,
                rmse_val
            ])

        out["F"] = np.array(F)

In [ ]:
pan, ms = load_images(
    "dataset/pan.tif",
    "dataset/ms.tif"
)

problem = FusionOptimization(pan, ms)

algorithm = NSGA2(
    pop_size=50
)

termination = get_termination("n_gen", 50)

result = minimize(
    problem,
    algorithm,
    termination,
    seed=1,
    save_history=True,
    verbose=True
)

In [ ]:
print("Best Solutions:")

for sol in result.X:

    alpha = sol[0]
    wavelet = wavelets[int(sol[1])]
    level = int(sol[2])

    print(alpha, wavelet, level)

In [ ]:
best = result.X[0]

alpha = best[0]
wavelet = wavelets[int(best[1])]
level = int(best[2])

fused = awp_fusion(pan, ms, wavelet, level, alpha)

cv2.imwrite("fused_output.tif", fused.astype(np.uint8))